# JASSIM AI — Google Colab GPU Server
دفتر تشغيل تجريبي لخادم JASSIM AI مع Ollama وقاعدة بيانات SQLite محفوظة في Google Drive ونفق HTTPS مؤقت.

تنبيه: Colab قد ينهي الجلسة المجانية أو يغيّر توفر GPU. خلية Watchdog تعيد تشغيل العمليات المتعطلة ما دامت الجلسة حية، لكنها لا تمنع إيقاف Colab. لا تشارك رابط النفق أو رمز API.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, subprocess, time, json, secrets, requests, re, threading

PROJECT = Path('/content/JASSIM-AI-Android')
DATA_DIR = Path('/content/drive/MyDrive/JASSIM_AI/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
print('Persistent database directory:', DATA_DIR)

## 1. تنزيل ملفات الخادم

In [ ]:
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/stevennpllk-crypto/-JASSIM-AI-Android.git', str(PROJECT)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only', 'origin', 'main'], check=True)
SERVER = PROJECT / 'backend' / 'server.py'
assert SERVER.exists(), 'backend/server.py is missing'
print('Server source ready:', SERVER)

## 2. إعداد الحماية والنموذج

In [ ]:
TOKEN_FILE = DATA_DIR / '.api_token'
if TOKEN_FILE.exists():
    API_TOKEN = TOKEN_FILE.read_text().strip()
else:
    API_TOKEN = secrets.token_urlsafe(48)
    TOKEN_FILE.write_text(API_TOKEN)
    try:
        TOKEN_FILE.chmod(0o600)
    except Exception:
        pass
print('Persistent private API token loaded from Google Drive; it is never printed automatically.')

os.environ['JASSIM_API_TOKEN'] = API_TOKEN
os.environ['JASSIM_CORS_ORIGIN'] = 'https://stevennpllk-crypto.github.io'
os.environ['JASSIM_HOST'] = '127.0.0.1'
os.environ['JASSIM_PORT'] = '8787'
os.environ['JASSIM_DATA_DIR'] = str(DATA_DIR)
os.environ['JASSIM_OLLAMA_URL'] = 'http://127.0.0.1:11434'
gpu_probe = subprocess.run(['bash', '-lc', 'nvidia-smi -L'], capture_output=True, text=True)
has_nvidia_gpu = gpu_probe.returncode == 0 and 'GPU' in gpu_probe.stdout
default_model = 'qwen3.5:9b' if has_nvidia_gpu else 'qwen3.5:4b'
os.environ['JASSIM_OLLAMA_MODEL'] = os.environ.get('JASSIM_OLLAMA_MODEL') or default_model
os.environ['JASSIM_OLLAMA_KEEP_ALIVE'] = '-1'
os.environ['JASSIM_NUM_CTX'] = os.environ.get('JASSIM_NUM_CTX', '8192')
os.environ['JASSIM_NUM_PREDICT'] = '3072'
os.environ['JASSIM_HISTORY_CHAR_BUDGET'] = '32000'
os.environ['JASSIM_REQUEST_TIMEOUT'] = '300'
os.environ['JASSIM_MAX_CONCURRENT'] = '1'
os.environ['JASSIM_QUEUE_TIMEOUT'] = '180'
MODEL = os.environ['JASSIM_OLLAMA_MODEL']
print('NVIDIA GPU detected:', has_nvidia_gpu)
print('Selected model:', MODEL)
print('9B is selected automatically when NVIDIA GPU is available; otherwise the 4B fallback is used.')

## 3. تثبيت وتشغيل Ollama وتنزيل النموذج

In [ ]:
install = subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, text=True, capture_output=True)
print(install.stdout[-3000:])
if install.returncode != 0:
    print(install.stderr[-3000:])
    raise RuntimeError('Ollama installation failed')

ollama_log = open('/content/ollama.log', 'w')
ollama_proc = subprocess.Popen(['ollama', 'serve'], stdout=ollama_log, stderr=subprocess.STDOUT)

for _ in range(45):
    try:
        requests.get('http://127.0.0.1:11434/api/tags', timeout=3).raise_for_status()
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('Ollama did not start. Check /content/ollama.log')

pull = subprocess.run(['ollama', 'pull', MODEL], text=True)
if pull.returncode != 0:
    raise RuntimeError('Model download failed: ' + MODEL)
print('Ollama and model are ready; keep_alive=-1 keeps the model loaded while Ollama runs.')

## 4. تشغيل API مع تخزين دائم للمحادثات

In [ ]:
api_log = open('/content/jassim-api.log', 'w')
api_proc = subprocess.Popen(
    ['python', str(SERVER)],
    cwd=str(PROJECT / 'backend'),
    env=os.environ.copy(),
    stdout=api_log,
    stderr=subprocess.STDOUT
)

for _ in range(45):
    try:
        health = requests.get('http://127.0.0.1:8787/ready', timeout=3)
        if health.ok and health.json().get('ready'):
            break
    except Exception:
        pass
    time.sleep(2)
else:
    print(Path('/content/jassim-api.log').read_text(errors='ignore')[-4000:])
    raise RuntimeError('JASSIM API/model readiness failed')

print(json.dumps(health.json(), ensure_ascii=False, indent=2))

## 5. اختبار إجابة حقيقية من النموذج

In [ ]:
headers = {'Authorization': 'Bearer ' + API_TOKEN, 'Content-Type': 'application/json'}
test = requests.post(
    'http://127.0.0.1:8787/api/chat',
    headers=headers,
    json={'message': 'عرّف بنفسك واشرح كيف تساعد المستخدم بالعربية.', 'conversation_id': 'colab-check'},
    timeout=300
)
print('HTTP:', test.status_code)
print(json.dumps(test.json(), ensure_ascii=False, indent=2))
test.raise_for_status()
assert test.json().get('engine') == 'ollama', 'The response did not come from the AI model'

## 6. إنشاء رابط HTTPS مؤقت

### إعداد الربط التلقائي مرة واحدة

قبل تشغيل خلية النفق، أضف في Google Colab من **Secrets** سراً باسم `JASSIM_GITHUB_TOKEN`.
أنشئ GitHub Fine-grained personal access token لحسابك، وحدد هذا المستودع فقط
`stevennpllk-crypto/-JASSIM-AI-Android`، وامنحه **Contents: Read and write**.
لا تضع الرمز في خلية أو ملف أو رسالة؛ احفظه مباشرة في Colab Secrets.
بعدها تنشر الخلية عنوان النفق الحالي في `dashboard/connection.json`، ويكتشفه تطبيق Android تلقائياً.
رمز API الخاص بـ JASSIM يبقى محفوظاً في Google Drive ويُعرض فقط من خلية الرمز عند الحاجة لإدخاله مرة واحدة في التطبيق.


In [ ]:
deb = '/content/cloudflared.deb'
subprocess.run([
    'bash', '-lc',
    f'wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb -O {deb} && dpkg -i {deb}'
], check=True)

cf_log = open('/content/cloudflared.log', 'w')
cf_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8787'],
    stdout=cf_log, stderr=subprocess.STDOUT, text=True
)
tunnel_url = None
for _ in range(60):
    time.sleep(2)
    log_text = Path('/content/cloudflared.log').read_text(errors='ignore')
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log_text)
    if match:
        tunnel_url = match.group(0)
        break
    if cf_proc.poll() is not None:
        break

if not tunnel_url:
    print(Path('/content/cloudflared.log').read_text(errors='ignore')[-4000:])
    raise RuntimeError('Could not obtain a Cloudflare tunnel URL')

print('JASSIM HTTPS URL:', tunnel_url)
print('Public health:', requests.get(tunnel_url + '/health', timeout=20).json())

# Publish the changing tunnel address to the stable discovery file on GitHub Pages.
# Add a GitHub fine-grained token as a Colab Secret named JASSIM_GITHUB_TOKEN
# with Contents: Read and write access to this repository.
from google.colab import userdata
import base64
GH_TOKEN = userdata.get('JASSIM_GITHUB_TOKEN')
if not GH_TOKEN:
    raise RuntimeError('Add the Colab Secret JASSIM_GITHUB_TOKEN (repository Contents: Read and write), then rerun this cell.')

GITHUB_REPO = 'stevennpllk-crypto/-JASSIM-AI-Android'
REGISTRY_PATH = 'dashboard/connection.json'
REGISTRY_API = f'https://api.github.com/repos/{GITHUB_REPO}/contents/{REGISTRY_PATH}'
GH_HEADERS = {
    'Authorization': 'Bearer ' + GH_TOKEN,
    'Accept': 'application/vnd.github+json',
    'X-GitHub-Api-Version': '2022-11-28'
}

def _publish_endpoint(url):
    payload = {
        'status': 'online',
        'url': url,
        'updated_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        'service': 'JASSIM AI'
    }
    current = requests.get(REGISTRY_API, headers=GH_HEADERS, params={'ref': 'main'}, timeout=20)
    sha = current.json().get('sha') if current.status_code == 200 else None
    encoded = base64.b64encode(json.dumps(payload, ensure_ascii=False).encode('utf-8')).decode('ascii')
    commit = {
        'message': 'Update live JASSIM AI endpoint',
        'content': encoded,
        'branch': 'main'
    }
    if sha:
        commit['sha'] = sha
    response = requests.put(REGISTRY_API, headers=GH_HEADERS, json=commit, timeout=30)
    response.raise_for_status()
    print('Published current endpoint to the stable discovery registry.')

_publish_endpoint(tunnel_url)


## 7. ربط التطبيق والموقع

In [ ]:
print('API URL:', tunnel_url)
print('Database folder in Drive:', DATA_DIR)
print('Android discovers the current HTTPS endpoint automatically. Enter the private API token once in Android Settings; never share it.')

## 8. مراقبة العمليات وإعادة تشغيلها تلقائياً (Watchdog)

شغّل الخلية التالية مرة واحدة بعد نجاح الخطوات السابقة. إذا تعطّل Ollama أو API أو النفق، تحاول الخلية إعادة تشغيله. إذا تغيّر رابط النفق ستطبعه من جديد ويجب تحديثه في التطبيق.

هذه الميزة لا تمنع Google Colab من إيقاف الجلسة أو سحب GPU. للاستمرارية الدائمة استخدم خادماً يعمل طوال الوقت وتعليمات DEPLOYMENT.md.

In [ ]:
# Watchdog works only while the Colab runtime is alive.
watchdog_stop = threading.Event()
watchdog_state = {'api_failures': 0, 'tunnel_failures': 0}

def _wait_ollama(timeout=90):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            requests.get('http://127.0.0.1:11434/api/tags', timeout=3).raise_for_status()
            return True
        except Exception:
            time.sleep(3)
    return False

def _start_api():
    global api_proc, api_log
    api_log = open('/content/jassim-api.log', 'a')
    api_proc = subprocess.Popen(
        ['python', str(SERVER)], cwd=str(PROJECT / 'backend'),
        env=os.environ.copy(), stdout=api_log, stderr=subprocess.STDOUT
    )

def _start_tunnel():
    global cf_proc, cf_log, tunnel_url
    cf_log = open('/content/cloudflared.log', 'w')
    cf_proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8787'],
        stdout=cf_log, stderr=subprocess.STDOUT, text=True
    )
    new_url = None
    for _ in range(30):
        time.sleep(2)
        log_text = Path('/content/cloudflared.log').read_text(errors='ignore')
        matches = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', log_text)
        if matches:
            new_url = matches[-1]
            break
        if cf_proc.poll() is not None:
            break
    if new_url:
        tunnel_url = new_url
        print('[watchdog] New HTTPS URL detected; publishing it for Android auto-discovery...')
        try:
            _publish_endpoint(tunnel_url)
        except Exception as exc:
            print('[watchdog] Could not publish discovery endpoint:', type(exc).__name__)
    else:
        print('[watchdog] Tunnel did not recover; inspect /content/cloudflared.log')

def _watch_services():
    global ollama_proc, ollama_log, api_proc, cf_proc
    while not watchdog_stop.wait(25):
        try:
            if ollama_proc.poll() is not None:
                print('[watchdog] Restarting Ollama...')
                ollama_log = open('/content/ollama.log', 'a')
                ollama_proc = subprocess.Popen(['ollama', 'serve'], stdout=ollama_log, stderr=subprocess.STDOUT)
                if not _wait_ollama():
                    print('[watchdog] Ollama did not recover; inspect /content/ollama.log')
                    continue

            if api_proc.poll() is not None:
                print('[watchdog] Restarting JASSIM API...')
                _start_api()
                time.sleep(3)

            try:
                ready = requests.get('http://127.0.0.1:8787/health', timeout=6)
                ready.raise_for_status()
                watchdog_state['api_failures'] = 0
            except Exception:
                watchdog_state['api_failures'] += 1
                if watchdog_state['api_failures'] >= 3:
                    print('[watchdog] API readiness failed repeatedly; restarting API...')
                    if api_proc.poll() is None:
                        api_proc.terminate()
                        try:
                            api_proc.wait(timeout=5)
                        except subprocess.TimeoutExpired:
                            api_proc.kill()
                    _start_api()
                    watchdog_state['api_failures'] = 0

            if cf_proc.poll() is not None:
                print('[watchdog] Restarting Cloudflare tunnel...')
                _start_tunnel()
            else:
                try:
                    requests.get(tunnel_url + '/health', timeout=8).raise_for_status()
                    watchdog_state['tunnel_failures'] = 0
                except Exception:
                    watchdog_state['tunnel_failures'] += 1
                    if watchdog_state['tunnel_failures'] >= 3:
                        print('[watchdog] Public tunnel failed repeatedly; replacing it...')
                        cf_proc.terminate()
                        try:
                            cf_proc.wait(timeout=5)
                        except subprocess.TimeoutExpired:
                            cf_proc.kill()
                        _start_tunnel()
                        watchdog_state['tunnel_failures'] = 0
        except Exception as exc:
            print('[watchdog] Monitor error:', type(exc).__name__)

watchdog_thread = threading.Thread(target=_watch_services, daemon=True)
watchdog_thread.start()
print('Watchdog active. Stop it with watchdog_stop.set()')

## 9. إظهار رمز الربط عند الحاجة فقط
لا تشغّل هذه الخلية إلا عندما تريد إدخال الرمز يدوياً في تطبيق Android. لا تشارك لقطة الشاشة أو مخرجات هذه الخلية، ولا تحفظ نسخة عامة من الدفتر بعد تشغيلها.

In [ ]:
from IPython.display import display, Markdown
display(Markdown('**رمز API الخاص — لا تشاركه:**'))
print(API_TOKEN)